In [1]:
import os

# Твой блок путей
USER_HOME = "/home/nshevtsova"
os.environ['HF_HOME'] = f"{USER_HOME}/.cache/huggingface"
os.environ['XDG_CACHE_HOME'] = f"{USER_HOME}/.cache"
os.environ['MPLCONFIGDIR'] = f"{USER_HOME}/.config/matplotlib"


# Создаем папки, чтобы не было Permission Denied
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)


In [2]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

from diffusers import StableDiffusionPipeline
from torchvision import transforms

from torchmetrics.image.fid import FrechetInceptionDistance
from prdc import compute_prdc
import open_clip
import random


/opt/conda/envs/lora/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/envs/lora/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [3]:
import torch
import os
from diffusers import StableDiffusion3Pipeline
from transformers import CLIPTextModelWithProjection, T5EncoderModel, CLIPTokenizer, T5TokenizerFast

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None
).to("cuda")

pipe.load_lora_weights(
    "lora_output/1.5_prodigy",
    weight_name="last.safetensors"
)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()


Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  3.09it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


In [4]:
METADATA = Path("/home/nshevtsova/metadata_clean.csv")
IMG_DIR = Path("/home/nshevtsova/BCN_clean")

df = pd.read_csv(METADATA)


In [5]:
def generate_images(pipe, cls, n):

    prompt = f"{PROMPT_MAP_NEW[cls]} high resolution dermoscopy image"
    negative = "clinical photo, ruler, text, watermark, low quality"

    images = []

    for _ in range(n):
        with torch.no_grad(), torch.cuda.amp.autocast():
            img = pipe(
                prompt,
                negative_prompt=negative,
                num_images_per_prompt=1,
                guidance_scale=8.5,     
                num_inference_steps=30,    
                cross_attention_kwargs={"scale": 0.8}
            ).images[0]

        images.append(img)

    return images

In [6]:
def load_real_images(df, image_dir, cls, n=5):

    df_cls = df[df["class"] == cls]

    if len(df_cls) == 0:
        return []

    df_cls = df_cls.sample(n=min(n, len(df_cls)), random_state=42)

    images = []

    for isic_id in df_cls["isic_id"]:
        img_path = Path(image_dir) / f"{isic_id}.jpg"

        if img_path.exists():
            images.append(Image.open(img_path).convert("RGB"))

    print(f"Loaded images → real: {len(images)}")

    return images

In [7]:
fid = FrechetInceptionDistance(feature=2048).to("cpu")

def pil_to_uint8_tensor(img):
    # PIL → uint8 tensor [3,H,W]
    x = np.array(img, dtype=np.uint8)
    x = torch.from_numpy(x).permute(2, 0, 1)  # HWC → CHW
    return x

def compute_fid(real_imgs, fake_imgs):
    fid_metric = FrechetInceptionDistance(feature=2048).to(device)

    for img in real_imgs:
        x = pil_to_uint8_tensor(img).unsqueeze(0).to(device)
        fid_metric.update(x, real=True)

    for img in fake_imgs:
        x = pil_to_uint8_tensor(img).unsqueeze(0).to(device)
        fid_metric.update(x, real=False)

    fid_value = fid_metric.compute().item()
    return fid_value

In [8]:
import torchvision.models as models

device = torch.device("cpu")

inception = models.inception_v3(
    weights=models.Inception_V3_Weights.IMAGENET1K_V1,
    transform_input=False
).to("cpu").eval()

inception.fc = torch.nn.Identity()

def extract_features(images):
    feats = []
    preprocess = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor()
    ])

    with torch.no_grad():
        for img in images:
            x = preprocess(img).unsqueeze(0)
            f = inception(x)[0].detach().cpu().numpy()
            feats.append(f)

    return np.vstack(feats)


In [9]:
def compute_prd(real_imgs, fake_imgs):
    real_feats = extract_features(real_imgs)
    fake_feats = extract_features(fake_imgs)

    return compute_prdc(
        real_features=real_feats,
        fake_features=fake_feats,
        nearest_k=5
    )


In [10]:
MY_CACHE_DIR = os.path.join(USER_HOME, ".cache", "clip")
os.makedirs(MY_CACHE_DIR, exist_ok=True)

device = torch.device("cpu")

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai",
    cache_dir=MY_CACHE_DIR  
)

clip_model = clip_model.to(device).eval()
tokenizer = open_clip.get_tokenizer("ViT-B-32")


100%|███████████████████████████████████████| 354M/354M [00:07<00:00, 47.7MiB/s]


In [11]:
def clip_similarity(images, text):
    text_tokens = tokenizer([text])

    sims = []
    with torch.no_grad():
        text_feat = clip_model.encode_text(text_tokens).float()
        text_feat /= text_feat.norm(dim=-1, keepdim=True)

        for img in images:
            img_tensor = clip_preprocess(img).unsqueeze(0)
            img_feat = clip_model.encode_image(img_tensor).float()
            img_feat /= img_feat.norm(dim=-1, keepdim=True)

            sims.append((img_feat @ text_feat.T).item())

    return np.mean(sims)


In [12]:
MIN_REAL = 100
N_GEN = 100

results = []

DIAGNOSIS_MAP = {
    "Solar or actinic keratosis": "AK",
    "Basal cell carcinoma": "BCC",
    "Seborrheic keratosis": "BKL",
    "Solar lentigo": "BKL",
    "Dermatofibroma": "DF",
    "Melanoma metastasis": "MEL",
    "Melanoma, NOS": "MEL",
    "Nevus": "NV",
    "Squamous cell carcinoma, NOS": "SCC",
}

PROMPT_MAP = {
    "AK": "Actinic keratosis",
    "BCC": "Basal cell carcinoma",
    "BKL": "Benign keratosis lesion",
    "DF": "Dermatofibroma",
    "MEL": "Melanoma",
    "NV": "Nevus",
    "SCC": "Squamous cell carcinoma",
}

PROMPT_MAP_NEW = {
    "AK": "<ak_skinlesion> actinic keratosis skin lesion with rough scaly surface",
    "BCC": "<bcc_skinlesion> basal cell carcinoma skin cancer with pearly border and visible blood vessels",
    "BKL": "<bkl_skinlesion> benign keratosis skin lesion with waxy stuck-on appearance", 
    "DF": "<df_skinlesion>v dermatofibroma benign skin lesion with firm structure and central white scar-like area", 
    "MEL": "<mel_skinlesion> malignant melanoma skin cancer with asymmetric shape irregular border and multiple colors", 
    "NV": "<nv_skinlesion> benign melanocytic nevus mole with symmetric round shape and uniform brown pigmentation",
    "SCC": "<scc_skinlesion> squamous cell carcinoma skin cancer with scaly surface and irregular structure",
}

df = pd.read_csv(METADATA)
df["class"] = df["diagnosis_3"].map(DIAGNOSIS_MAP)

# удаляем все строки без класса
df = df.dropna(subset=["class"])
classes = sorted(df["class"].unique())

print(f"Found {len(classes)} classes: {classes}")

for cls in classes:

    print(f"\n=== Class {cls} ===")

    real_imgs = load_real_images(
        df=df,
        image_dir=IMG_DIR,
        cls=cls,
        n=N_GEN
    )

    if len(real_imgs) < 2:
        print(f"[SKIP] Not enough real images for {cls}")
        continue

    fake_imgs = generate_images(
        pipe,
        cls,
        n=N_GEN
    )

    fid = compute_fid(real_imgs, fake_imgs)

    prd = compute_prd(real_imgs, fake_imgs)

    clip_sim = clip_similarity(
        fake_imgs,
        f"dermoscopy image of {PROMPT_MAP[cls]}"
    )

    results.append({
        "class": cls,
        "n_real": len(real_imgs),
        "fid": fid,
        "precision": prd["precision"],
        "recall": prd["recall"],
        "clip": clip_sim,
    })

    print(f"FID: {fid:.2f}")
    print(f"Precision: {prd['precision']:.3f}")
    print(f"Recall: {prd['recall']:.3f}")
    print(f"CLIP: {clip_sim:.3f}")

    del fake_imgs
    torch.cuda.empty_cache()


Found 7 classes: ['AK', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCC']

=== Class AK ===
Loaded images → real: 100


/tmp/ipykernel_5793/3201848000.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():
100%|██████████| 30/30 [00:04<00:00,  6.31it/s]


Num real: 100 Num fake: 100
FID: 161.46
Precision: 0.800
Recall: 0.490
CLIP: 0.310

=== Class BCC ===


/tmp/ipykernel_5793/3201848000.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


Loaded images → real: 100


...

Num real: 100 Num fake: 100
FID: 233.87
Precision: 0.910
Recall: 0.140
CLIP: 0.289

=== Class NV ===


/tmp/ipykernel_5793/3201848000.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


Loaded images → real: 100


 13%|█▎        | 4/30 [00:00<00:04,  5.28it/s]


KeyboardInterrupt: 